In [36]:
import numpy as np
import pandas as pd

In [41]:
proxy = pd.read_csv("../proxy/realized_cov_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()
ewma  = pd.read_csv("../models/ewma_cov_lambda094.csv", parse_dates=["Date"]).set_index("Date").sort_index()
dcc   = pd.read_csv("../results/dcc_cov_forecast_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()
har = pd.read_csv("../results/har_cov_forecast_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()

In [46]:
def make_pd(mat, eps=1e-8):
    mat = 0.5 * (mat + mat.T)

    eigvals, eigvecs = np.linalg.eigh(mat)

    eigvals[eigvals < eps] = eps

    mat_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    mat_pd = 0.5 * (mat_pd + mat_pd.T)

    return mat_pd

def project_df_to_pd(df, n_assets=8):

    rows = []

    for dt in df.index:

        mat = row_to_matrix(df.loc[dt], n_assets=n_assets)

        mat_pd = make_pd(mat)

        rows.append(mat_pd.reshape(-1))

    df_pd = pd.DataFrame(rows, index=df.index, columns=df.columns)

    return df_pd

har = project_df_to_pd(har)

In [61]:
har_losses.sort_values("QLIKE", ascending=False).head(10)

proxy_var = proxy.apply(lambda r: np.trace(row_to_matrix(r)), axis=1)
har_var   = har.apply(lambda r: np.trace(row_to_matrix(r)), axis=1)

vol_compare = pd.DataFrame({
    "proxy": proxy_var,
    "har": har_var
}).loc[har_losses.index]

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=vol_compare.index,
    y=vol_compare["proxy"],
    mode="lines",
    name="Realized"
))

fig.add_trace(go.Scatter(
    x=vol_compare.index,
    y=vol_compare["har"],
    mode="lines",
    name="HAR forecast"
))

fig.update_layout(
    title="Variance scale: Realized vs HAR",
    xaxis_title="Date",
    yaxis_title="Trace of covariance",
    hovermode="x unified"
)

fig.show()

def qlike_terms(proxy_mat, forecast_mat):
    H_inv = np.linalg.inv(forecast_mat)
    trace_term = np.trace(proxy_mat @ H_inv)
    logdet_term = np.linalg.slogdet(proxy_mat)[1] - np.linalg.slogdet(forecast_mat)[1]
    return trace_term, logdet_term

dt = har_losses["QLIKE"].idxmax()

S = row_to_matrix(proxy.loc[dt])
H = row_to_matrix(har.loc[dt])

trace_term, logdet_term = qlike_terms(S, H)

print("trace term:", trace_term)
print("logdet term:", logdet_term)
print("qlike:", trace_term - logdet_term - S.shape[0])

trace term: 2314.4869492084986
logdet term: 1.4919787104438171
qlike: 2304.994970498055


In [48]:
print("Proxy shape:", proxy.shape)
print("EWMA shape:", ewma.shape)
print("DCC shape:", dcc.shape)
print("HAR shape:", har.shape)

print("Proxy vs EWMA same columns:", list(proxy.columns) == list(ewma.columns))
print("Proxy vs DCC same columns:", list(proxy.columns) == list(dcc.columns))
print("Proxy vs HAR same columns:", list(proxy.columns) == list(har.columns))


common_ewma = proxy.index.intersection(ewma.index)
common_dcc  = proxy.index.intersection(dcc.index)
common_har  = proxy.index.intersection(har.index)

print("EWMA common dates:", len(common_ewma), common_ewma.min().date(), "->", common_ewma.max().date())
print("DCC common dates:", len(common_dcc), common_dcc.min().date(), "->", common_dcc.max().date())
print("HAR common dates:", len(common_har), common_har.min().date(), "->", common_har.max().date())

Proxy shape: (2184, 64)
EWMA shape: (1255, 64)
DCC shape: (1255, 64)
HAR shape: (1235, 64)
Proxy vs EWMA same columns: True
Proxy vs DCC same columns: True
Proxy vs HAR same columns: True
EWMA common dates: 1235 2021-01-04 -> 2025-12-02
DCC common dates: 1235 2021-01-04 -> 2025-12-02
HAR common dates: 1235 2021-01-04 -> 2025-12-02


In [49]:
def row_to_matrix(row, n_assets=8):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat

def make_spd(mat, eps=1e-10):
    mat = 0.5 * (mat + mat.T)
    mat = mat + eps * np.eye(mat.shape[0])
    return mat

def qlike_loss(proxy_mat, forecast_mat, eps=1e-10):
    S = make_spd(proxy_mat, eps=eps)
    H = make_spd(forecast_mat, eps=eps)

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)

def frobenius_loss(proxy_mat, forecast_mat):
    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):
    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "QLIKE": qlike_loss(S_mat, H_mat),
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean()
    })

    return losses, summary

In [50]:
ewma_losses, ewma_summary = evaluate_against_proxy(ewma, proxy, model_name="EWMA")
dcc_losses, dcc_summary = evaluate_against_proxy(dcc, proxy, model_name="DCC")
har_losses, har_summary = evaluate_against_proxy(har, proxy, model_name="HAR")

summary_table = pd.DataFrame([ewma_summary, dcc_summary,har_summary]).reset_index(drop=True)
summary_table

,model,n_obs,start_date,end_date,avg_qlike,avg_frobenius
0,EWMA,1235,2021-01-04,2025-12-02,5.235591,0.000002
1,DCC,1235,2021-01-04,2025-12-02,4.488435,0.000017
2,HAR,1235,2021-01-04,2025-12-02,48.226165,0.000002


In [51]:
loss_compare = pd.DataFrame({
    "EWMA_QLIKE": ewma_losses["QLIKE"],
    "DCC_QLIKE": dcc_losses["QLIKE"],
    "HAR_QLIKE": har_losses["QLIKE"],
    "EWMA_FROB": ewma_losses["FROBENIUS"],
    "DCC_FROB": dcc_losses["FROBENIUS"],
    "HAR_FROB": har_losses["FROBENIUS"],
})

loss_compare.describe()

qlike_compare = pd.DataFrame({
    "EWMA": ewma_losses["QLIKE"],
    "DCC": dcc_losses["QLIKE"],
    "HAR": har_losses["QLIKE"]
})

print("Best model by QLIKE share:")
print(qlike_compare.idxmin(axis=1).value_counts(normalize=True))

Best model by QLIKE share:
DCC     0.436437
HAR     0.379757
EWMA    0.183806
Name: proportion, dtype: float64


Varians and correlation performance


In [52]:
def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat, eps=1e-12):
    cov_mat = 0.5 * (cov_mat + cov_mat.T)

    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, eps))

    corr = cov_mat / np.outer(std, std)

    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr

In [53]:
def variance_mse_loss(proxy_mat, forecast_mat):

    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)

    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):

    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]

    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]

    return float(np.mean(diff ** 2))

In [54]:
def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary

In [55]:
ewma_comp_losses, ewma_comp_summary = evaluate_components_against_proxy(
    ewma, proxy, model_name="EWMA"
)

dcc_comp_losses, dcc_comp_summary = evaluate_components_against_proxy(
    dcc, proxy, model_name="DCC"
)

har_comp_losses, har_comp_summary = evaluate_components_against_proxy(
    har, proxy, model_name="HAR"
)

component_summary_table = pd.DataFrame(
    [ewma_comp_summary, dcc_comp_summary, har_comp_summary]
).reset_index(drop=True)

component_summary_table

,model,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
0,EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
1,DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739
2,HAR,1235,2021-01-04,2025-12-02,1.440989e-07,0.070277


Høy/lav volatilitet 

In [56]:
proxy_test = proxy.loc[ewma_losses.index]

proxy_vol = proxy_test.apply(
    lambda row: np.trace(row_to_matrix(row)), axis=1
)

proxy_vol.name = "system_volatility"

proxy_vol.describe()

count    1235.000000
mean        0.001927
std         0.001188
min         0.000348
25%         0.001117
50%         0.001560
75%         0.002403
max         0.006489
Name: system_volatility, dtype: float64

In [57]:
low_threshold = proxy_vol.quantile(0.25)
high_threshold = proxy_vol.quantile(0.75)

print("Low volatility threshold:", low_threshold)
print("High volatility threshold:", high_threshold)

vol_regime = pd.Series(index=proxy_vol.index, dtype="object")

vol_regime[proxy_vol <= low_threshold] = "low"
vol_regime[proxy_vol >= high_threshold] = "high"
vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

vol_regime.value_counts()

Low volatility threshold: 0.001117003960638956
High volatility threshold: 0.0024032604891289543


mid     617
high    309
low     309
Name: count, dtype: int64

In [58]:
ewma_losses["regime"] = vol_regime.loc[ewma_losses.index]
dcc_losses["regime"] = vol_regime.loc[dcc_losses.index]
har_losses["regime"] = vol_regime.loc[har_losses.index]

ewma_comp_losses["regime"] = vol_regime.loc[ewma_comp_losses.index]
dcc_comp_losses["regime"] = vol_regime.loc[dcc_comp_losses.index]
har_comp_losses["regime"] = vol_regime.loc[har_comp_losses.index]


qlike_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["QLIKE"].mean(),
    "DCC": dcc_losses.groupby("regime")["QLIKE"].mean(),
    "HAR": har_losses.groupby("regime")["QLIKE"].mean()
})

qlike_regime_table

frob_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["FROBENIUS"].mean(),
    "DCC": dcc_losses.groupby("regime")["FROBENIUS"].mean(),
    "HAR": har_losses.groupby("regime")["FROBENIUS"].mean()
})

frob_regime_table

var_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["VAR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["VAR_MSE"].mean(),
    "HAR": har_comp_losses.groupby("regime")["VAR_MSE"].mean()
})

var_regime_table

corr_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["CORR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["CORR_MSE"].mean(),
    "HAR": har_comp_losses.groupby("regime")["CORR_MSE"].mean()
})

corr_regime_table

,EWMA,DCC,HAR
regime,,,
high,0.101057,0.096769,0.088949
low,0.073199,0.075213,0.058710
mid,0.073534,0.065470,0.066718


In [59]:
regime_summary = pd.concat(
    {
        "QLIKE": qlike_regime_table,
        "FROBENIUS": frob_regime_table,
        "VAR_MSE": var_regime_table,
        "CORR_MSE": corr_regime_table
    },
    axis=1
)

regime_summary

QLIKE                           FROBENIUS                      \
            EWMA       DCC         HAR          EWMA       DCC       HAR   
regime                                                                     
high    7.515286  6.111136    8.314505  3.213900e-06  0.000006  0.000002   
low     4.218387  4.214621  103.112932  6.027697e-07  0.000048  0.000002   
mid     4.603322  3.812899   40.726476  1.211043e-06  0.000008  0.000001   

             VAR_MSE                              CORR_MSE                      
                EWMA           DCC           HAR      EWMA       DCC       HAR  
regime                                                                          
high    3.113609e-07  6.091462e-07  1.694685e-07  0.101057  0.096769  0.088949  
low     5.092294e-08  5.976040e-06  1.798662e-07  0.073199  0.075213  0.058710  
mid     1.171445e-07  9.234530e-07  1.134810e-07  0.073534  0.065470  0.066718

In [60]:
def system_volatility(df, n_assets=8):
    vol = df.apply(
        lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
        axis=1
    )
    return vol

realized_vol = system_volatility(proxy)
ewma_vol = system_volatility(ewma)
dcc_vol = system_volatility(dcc)
har_vol = system_volatility(har)

vol_df = pd.DataFrame({
    "Realized": realized_vol,
    "EWMA": ewma_vol,
    "DCC": dcc_vol,
    "HAR": har_vol
}).dropna()

vol_df.head()

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["Realized"],
    mode="lines",
    name="Realized volatility",
    line=dict(width=2)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["EWMA"],
    mode="lines",
    name="EWMA forecast",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["DCC"],
    mode="lines",
    name="DCC forecast",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["HAR"],
    mode="lines",
    name="HAR forecast",
    line=dict(width=1)
))

fig.update_layout(
    title="System Volatility: Realized vs Forecast",
    xaxis_title="Date",
    yaxis_title="Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()